<a href="https://colab.research.google.com/github/Karthik0484/Chat-Application/blob/main/Summarizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install --upgrade youtube-transcript-api


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.0/485.0 kB 8.5 MB/s eta 0:00:00


In [2]:
from youtube_transcript_api import YouTubeTranscriptApi


In [3]:
try:
    from youtube_transcript_api.formatters import TextFormatter
    _HAS_FORMATTER = True
except Exception:
    _HAS_FORMATTER = False


In [4]:
def extract_video_id(url):
    from urllib.parse import urlparse, parse_qs
    import re
    parsed = urlparse(url)
    if parsed.netloc.endswith("youtu.be"):
        return parsed.path.lstrip("/").split("?")[0]
    if parsed.path.startswith("/shorts/"):
        return parsed.path.split("/")[2].split("?")[0]
    qs = parse_qs(parsed.query)
    if "v" in qs:
        return qs["v"][0]
    m = re.search(r"([A-Za-z0-9_-]{11})", url)
    return m.group(1) if m else None

video_url = "https://youtu.be/K3bPO9CPQy4?si=UzNnqDfB6I2L1rGr"
video_id = extract_video_id(video_url)

api = YouTubeTranscriptApi()
transcript = api.fetch(video_id)   # this is what worked for you

if _HAS_FORMATTER:
    text = TextFormatter().format_transcript(transcript)
else:
    text = "\n".join(seg["text"] for seg in transcript)

print("\n--- TRANSCRIPT START ---\n")

print(text)

print("\n--- TRANSCRIPT End ---\n")


--- TRANSCRIPT START ---

friends if you're using chat GPT or any
other AI Tools in your research then
definitely it will simplify your
research writing but you have to be very
careful because at the same time it will
increase the possibilities of your
documents getting caught under this AI
detection tools that does mean your
documents can show 100% AI score and if
you're submitting these documents to
journals or universities then you may
face several consequences if you want to
avoid that then you have to follow the
techniques that I'll be mentioning here
in this video very carefully so hello
everyone welcome you all to this channel
my resar support and today in this video
we'll be discussing about three
important techniques those you have to
follow and then even if you're using
chat JP then also your research
documents may not show any air score
percentage most preferably turn it into
a detection tool and this is really the
most important thing there are many AI
tools available in t

In [11]:
import os
os.environ["TOGETHER_API_KEY"] = "ed99be2c320bb410f3eb8d6d5aa77062c66f57243bc34158148f081df3291824"


In [17]:
import youtube_transcript_api
from youtube_transcript_api import YouTubeTranscriptApi
import inspect
print('module file:', youtube_transcript_api.__file__)
print('attrs:', [a for a in dir(YouTubeTranscriptApi) if not a.startswith('_')])


module file: /usr/local/lib/python3.11/dist-packages/youtube_transcript_api/__init__.py
attrs: ['fetch', 'list']


In [23]:
import os
import re
from urllib.parse import urlparse, parse_qs

# youtube_transcript_api imports (be defensive about exceptions)
from youtube_transcript_api import YouTubeTranscriptApi
try:
    from youtube_transcript_api import NoTranscriptFound, TranscriptsDisabled
except Exception:
    NoTranscriptFound = Exception
    TranscriptsDisabled = Exception

import together

# ====================
# 1. Extract Video ID
# ====================
def extract_video_id(url):
    parsed = urlparse(url)
    if parsed.netloc.endswith("youtu.be"):
        return parsed.path.lstrip("/").split("?")[0]
    if parsed.path.startswith("/shorts/"):
        return parsed.path.split("/")[2].split("?")[0]
    qs = parse_qs(parsed.query)
    if "v" in qs:
        return qs["v"][0]
    m = re.search(r"([A-Za-z0-9_-]{11})", url)
    return m.group(1) if m else None


# ====================
# 2. Robust get_transcript
# ====================
def get_transcript(video_id, languages=['en'], debug=False):
    """
    Try multiple youtube_transcript_api shapes and return transcript text or None.
    """
    try:
        # 1) Try YouTubeTranscriptApi.get_transcript (some installs expose this)
        if hasattr(YouTubeTranscriptApi, "get_transcript"):
            if debug: print("Using YouTubeTranscriptApi.get_transcript(...)")
            data = YouTubeTranscriptApi.get_transcript(video_id, languages=languages)
            return " ".join([seg.get("text", "") for seg in data])

        # 2) Try class-level list() (if it's a classmethod in this install)
        try:
            if debug: print("Trying class call: YouTubeTranscriptApi.list(video_id)")
            tl = YouTubeTranscriptApi.list(video_id)  # may raise TypeError if list is instance method
        except TypeError:
            if debug: print("Class call failed; using instance: YouTubeTranscriptApi().list(video_id)")
            tl = YouTubeTranscriptApi().list(video_id)

        if debug:
            print("Transcript list type:", type(tl))
            # print(dir(tl))  # uncomment for deeper debugging

        # If tl has find_transcript (TranscriptList-like)
        if hasattr(tl, "find_transcript"):
            try:
                transcript_obj = tl.find_transcript(languages)
            except Exception:
                transcript_obj = next(iter(tl))
            if hasattr(transcript_obj, "fetch"):
                data = transcript_obj.fetch()
            else:
                data = transcript_obj

        # If tl itself has fetch()
        elif hasattr(tl, "fetch"):
            data = tl.fetch()

        # If tl is a plain list/tuple
        elif isinstance(tl, (list, tuple)):
            if len(tl) == 0:
                raise NoTranscriptFound(video_id)
            first = tl[0]
            if isinstance(first, dict) and "text" in first:
                data = tl
            elif hasattr(first, "fetch"):
                data = first.fetch()
            else:
                # try to coerce elements
                data = []
                for item in tl:
                    if isinstance(item, dict) and "text" in item:
                        data.append(item)
                    elif hasattr(item, "fetch"):
                        data = item.fetch()
                        break
                if not data:
                    raise RuntimeError("Unable to interpret transcript list elements.")
        else:
            raise RuntimeError(f"Unknown transcript list shape: {type(tl)}")

        # Validate and join
        if not data or not isinstance(data, (list, tuple)) or not isinstance(data[0], dict) or "text" not in data[0]:
            raise RuntimeError("Transcript fetch returned unexpected shape.")
        return " ".join([seg.get("text", "") for seg in data])

    except (NoTranscriptFound, TranscriptsDisabled) as e:
        if debug: print("Transcript not available:", e)
        return None
    except Exception as e:
        if debug: print("Error fetching transcript:", e)
        return None


# ====================
# 3. Ask AI About Video (keeps your Together.ai code)
# ====================
def ask_about_video(transcript, question):
    api_key = os.environ.get("TOGETHER_API_KEY")
    if not api_key:
        raise ValueError("TOGETHER_API_KEY environment variable not set")

    together.api_key = api_key

    prompt = f"""
You are an AI assistant. Below is the transcript of a YouTube video.

Transcript:
{transcript}

Question: {question}

Answer the question using only the content of the transcript.
    """
    try:
        resp = together.complete.create(
            model="meta-llama/Llama-3-70b-chat-hf",
            prompt=prompt,
            max_tokens=500,
            temperature=0.3
        )
        # adjust depending on Together's response shape
        if hasattr(resp, "choices") and len(resp.choices) > 0:
            return getattr(resp.choices[0], "text", resp.choices[0])
        return str(resp)
    except Exception as e:
        print(f"❌ Error calling Together AI: {e}")
        return "Could not get an answer from the AI model."


# ====================
# 4. Main Flow
# ====================
if __name__ == "__main__":
    video_url = "https://youtu.be/z6V53lKjiiI?si=125GXCljayJgbxKL"
    video_id = extract_video_id(video_url)

    if not video_id:
        print(f"❌ Could not extract video ID from the URL: {video_url}")
    else:
        print(f"Extracted Video ID: {video_id}")
        # Set debug=True to see internal shapes/diagnostics
        transcript = get_transcript(video_id, debug=True)

        if transcript:
            print("\n--- Transcript snippet ---\n")
            print(transcript[:500] + "...")  # Print a snippet

            if "TOGETHER_API_KEY" not in os.environ:
                os.environ["TOGETHER_API_KEY"] = input("Enter your Together API key: ").strip()

            user_question = input("\nAsk a question about the video: ")
            answer = ask_about_video(transcript, user_question)

            print("\n--- AI Answer ---\n")
            print(answer)
        else:
            print("Cannot answer questions without a transcript.")


Extracted Video ID: z6V53lKjiiI
Trying class call: YouTubeTranscriptApi.list(video_id)
Class call failed; using instance: YouTubeTranscriptApi().list(video_id)
Transcript list type: <class 'youtube_transcript_api._transcripts.TranscriptList'>
Error fetching transcript: Transcript fetch returned unexpected shape.
Cannot answer questions without a transcript.


In [7]:
import os
import re
from urllib.parse import urlparse, parse_qs
from youtube_transcript_api import YouTubeTranscriptApi
import together

# ====================
# 1. Extract Video ID
# ====================
def extract_video_id(url):
    parsed = urlparse(url)
    if parsed.netloc.endswith("youtu.be"):
        return parsed.path.lstrip("/").split("?")[0]
    if parsed.path.startswith("/shorts/"):
        return parsed.path.split("/")[2].split("?")[0]
    qs = parse_qs(parsed.query)
    if "v" in qs:
        return qs["v"][0]
    m = re.search(r"([A-Za-z0-9_-]{11})", url)
    return m.group(1) if m else None

# ====================
# 2. Get Transcript (using your working method)
# ====================
def get_transcript(video_id):
    try:
        api = YouTubeTranscriptApi()
        transcript_data = api.fetch(video_id)

        # transcript_data might be a list of dicts or list of objects with .text attribute
        transcript_texts = []
        for segment in transcript_data:
            # Support both dict and object types
            if isinstance(segment, dict):
                transcript_texts.append(segment.get('text', ''))
            else:
                # fallback for objects with .text attribute
                transcript_texts.append(getattr(segment, 'text', ''))

        transcript_text = " ".join(transcript_texts)
        print("✅ Transcript fetched successfully.")
        return transcript_text

    except Exception as e:
        print(f"⚠️ Could not fetch transcript: {e}")
        return None


# ====================
# 3. Ask AI About Video
# ====================
def ask_about_video(transcript, question):
    api_key = os.environ.get("TOGETHER_API_KEY")
    if not api_key:
        raise ValueError("TOGETHER_API_KEY environment variable not set")

    together.api_key = api_key

    prompt = f"""
You are an AI assistant. Below is the transcript of a YouTube video.

Transcript:
{transcript}

Question: {question}

Answer the question using only the content of the transcript.
    """
    try:
        resp = together.Completion.create(
            model="meta-llama/Llama-3-70b-chat-hf",
            prompt=prompt,
            max_tokens=500,
            temperature=0.3
        )
        # `resp.choices` might be a list of objects with 'text' attribute
        return resp.choices[0].text
    except Exception as e:
        print(f"❌ Error calling Together AI: {e}")
        return "Could not get an answer from the AI model."


# ====================
# 4. Main Flow
# ====================
if __name__ == "__main__":
    video_url = input("Enter YouTube video URL: ").strip()
    video_id = extract_video_id(video_url)

    if not video_id:
        print(f"❌ Could not extract video ID from the URL: {video_url}")
    else:
        print(f"Extracted Video ID: {video_id}")
        transcript = get_transcript(video_id)

        if transcript:
            print("\n--- Transcript snippet ---\n")
            print(transcript[:500] + "...")  # Print a snippet

            if "TOGETHER_API_KEY" not in os.environ:
                os.environ["TOGETHER_API_KEY"] = input("Enter your Together API key: ").strip()

            user_question = input("\nAsk a question about the video: ")
            answer = ask_about_video(transcript, user_question)

            print("\n--- AI Answer ---\n")
            print(answer)
        else:
            print("Cannot answer questions without a transcript.")


Enter YouTube video URL: https://youtu.be/z6V53lKjiiI?si=D1tgbkIDYynzOGWG
Extracted Video ID: z6V53lKjiiI
✅ Transcript fetched successfully.

--- Transcript snippet ---

when I was doing my research there weren't a lot of AI tools available or at least people didn't know about it so I had to spend a lot of my time doing literature survey managing my documents writing the research paper and then getting it plagiarism checked but nowadays there are so many AI tools that are available which you can use to make your life easy and put up the best research possible [Music] hi everyone I am Niha agrawal I'm the founder of eyes of communications and in this video we are...

Ask a question about the video: explain what is ai


/tmp/ipython-input-644924163.py:70: DeprecationWarning: Call to deprecated function create.
  resp = together.Completion.create(



--- AI Answer ---

 AI stands for Artificial Intelligence. It refers to tools that can make tasks easier and more efficient. In the context of research, AI tools can be used to manage literature, summarize research papers, check for plagiarism, and even assist with writing and grammar correction. Examples of AI tools mentioned in the transcript include Research Rabbit, Consensus, ChatPDF, String Car, and Page.ai. These tools can help researchers with tasks such as finding new papers, understanding topics, clearing doubts, writing research papers, and checking for plagiarism.


In [2]:
!pip install youtube-transcript-api together

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.0/485.0 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.7/102.7 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 10.4 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.2.1
    Uninstalling click-8.2.1:
      Successfully uninstalled click-8.2.1
  Attempting uninstall: typer
    Found existing installation: typer 0.16.0
    Uninstalling typer-0.16.0:
      Successfully uninstalled typer-0.16.0
